# 20_model_eval — 모델 학습 & 성능 평가

**한 줄 요약:** 최종 데이터로 **fingerprint만 / descriptor만 / 둘 결합** 세 가지 입력을 각각 학습시켜 어느 게 좋은지 **점수로 비교**한다.

**핵심 용어**
- **모델(model)**: 입력(분자 특징)으로 정답(active/inactive)을 맞히도록 데이터에서 규칙을 배우는 것. 여기선 **RandomForest**(결정트리 여러 개의 투표).
- **교차검증(cross-validation)**: 데이터를 5조각으로 나눠 → 4조각으로 배우고 나머지 1조각을 맞혀보기를 5번 회전.
  이렇게 하면 모든 분자가 **'배울 때 안 본 상태'로 채점**돼서 공정하다.
- **지표**: MCC·Accuracy·Recall·Precision(혼동행렬 기반), ROC-AUC·PR-AUC(확률 순위 품질).

**큰 흐름:** ① 도구 → ② 데이터·3입력 준비 → ③ 채점 함수 → ④ 3입력 각각 학습·채점 → ⑤ 그래프

> ⚠️ 지금 inactive의 대부분이 decoy(구조가 일부러 다름)라 점수가 매우 높게(≈0.99) 나온다. 이는 '실력'이 아니라 '문제가 쉬워서'다.

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다.
> - 앞 셀에서 만든 **변수**(값에 붙인 이름표)를 뒤 셀에서 계속 쓴다. 그래서 **순서대로** 실행해야 한다.
> - 코드 줄 뒤의 `# ...` 은 **주석**(설명)이며 실행에 영향이 없다.
> - 자주 나오는 것: `=`(오른쪽 값을 왼쪽 이름에 저장), `[ ]`(리스트=순서 있는 목록), `{ }`(딕셔너리=이름표-값 쌍),
>   `for x in 목록:`(목록을 하나씩 꺼내 반복), `def 함수(입력):`(재사용 작업 묶음),
>   **DataFrame(df)** = 엑셀 표처럼 행·열이 있는 데이터(판다스). `df['열이름']`으로 한 열을 고른다.

### 셀 1 — 도구(라이브러리) 불러오기
- `RandomForestClassifier`: 학습에 쓸 모델.
- `StratifiedKFold`, `cross_val_predict`: 클래스 비율을 유지한 5조각 나누기 + 교차검증 예측.
- `matplotlib`: 그래프 그리기(`Agg`는 화면 없이 파일로 저장하는 모드).
- `sklearn.metrics`: 점수(AUC·MCC 등)와 곡선을 계산하는 함수들.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')            # 그래프를 파일로 저장하는 모드(화면 없어도 동작). 노트북에선 자동 표시됨
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier               # 모델: 랜덤포레스트(트리들의 투표)
from sklearn.model_selection import StratifiedKFold, cross_val_predict  # 5조각 교차검증
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             matthews_corrcoef, roc_curve, precision_recall_curve)  # 점수/곡선 계산

### 셀 2 — 데이터 읽고 '입력 3종류' 만들기
`y`는 정답(potency). 입력 열을 세 그룹으로 나눈다:
- `fp_cols`: 이름이 `fp_`로 시작하는 fingerprint 열 1024개
- `desc_cols`: SMILES·potency·fingerprint를 뺀 나머지 = descriptor 열
- `FEATURES`: **딕셔너리**로 '입력이름 → 입력표' 3개를 담아둔다 → 아래 셀에서 하나씩 꺼내 반복 평가.
`.to_numpy()`는 표(df)를 모델이 먹기 좋은 **숫자 배열**로 바꾼다.

In [ ]:
# 최종 학습데이터 로드 + 3가지 특징 집합 정의
SRC = 'data/HSD17B13_final_training_1to1.csv'
df = pd.read_csv(SRC)
y = df['potency'].to_numpy()                             # 정답(1=active/0=inactive) 배열

fp_cols = [c for c in df.columns if c.startswith('fp_')] # fingerprint 열 이름 1024개
non_desc = set(['canonical_smiles', 'potency'] + fp_cols) # descriptor가 '아닌' 열들
desc_cols = [c for c in df.columns if c not in non_desc] # 나머지 = descriptor 열
print('화합물', len(df), '| potency', dict(pd.Series(y).value_counts()))
print('fingerprint 열', len(fp_cols), '| descriptor 열', len(desc_cols))

FEATURES = {                                             # 비교할 입력 3종 (이름 → 숫자 표)
    'fingerprint':      df[fp_cols].to_numpy(),          # 구조 지문만
    'descriptor':       df[desc_cols].to_numpy(),        # 물성만
    '결합(FP+desc)':    df[fp_cols + desc_cols].to_numpy(),  # 둘 다 합침
}

### 셀 3 — 채점 함수 정의 (혼동행렬 기반)
모델이 낸 **확률(proba)**을 0.5 기준으로 0/1 **예측(pred)**으로 바꾼 뒤, 맞고 틀림을 4가지로 센다:
**TP**(진짜 active를 active로), **TN**(진짜 inactive를 inactive로), **FP**(inactive를 active로 오답), **FN**(active를 놓침).
이 네 값으로 지표를 계산한다 — Accuracy·Recall·Precision·MCC. ROC-AUC·PR-AUC는 확률 순위의 품질(임계값 무관).
`def ...: return dict(...)` = 계산 결과들을 이름표 붙여 한 묶음으로 돌려준다.

In [ ]:
# 성능 지표 함수 (논문 정의: 혼동행렬 TP/TN/FP/FN 기반)
def metrics_from(y_true, proba, thr=0.5):                # 정답, 예측확률, 임계값을 받아 지표 반환
    pred = (proba >= thr).astype(int)                    # 확률 ≥ 0.5 → 1(active), 아니면 0
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()   # 혼동행렬 4요소로 분해
    acc = (tp + tn) / (tp + tn + fp + fn)                 # (2) Accuracy: 전체 중 맞힌 비율
    rec = tp / (tp + fn) if (tp + fn) else 0.0            # (3) Recall: 진짜 active를 얼마나 잡나
    prec = tp / (tp + fp) if (tp + fp) else 0.0           # (4) Precision: active라 한 것 중 진짜 비율
    mcc = matthews_corrcoef(y_true, pred)                 # (1) MCC: 네 값 모두 반영(불균형에 강함)
    roc = roc_auc_score(y_true, proba)                    # ROC-AUC
    pr = average_precision_score(y_true, proba)           # PR-AUC
    return dict(ROC_AUC=roc, PR_AUC=pr, MCC=mcc, Accuracy=acc,
               Recall=rec, Precision=prec, TP=tp, TN=tn, FP=fp, FN=fn)

### 셀 4 — 세 입력을 같은 방식으로 학습·채점
- `make_model()`: 모델을 새로 만들어 주는 함수. 괄호 안 `n_estimators=300`(트리 300개) 등이 **하이퍼파라미터**(사람이 정하는 설정 — 성능을 높이려면 여기를 튜닝).
- `for name, X in FEATURES.items()`: 입력 3종을 하나씩 꺼내(`name`=이름, `X`=표)
  - `cross_val_predict(...)`: 5조각 교차검증으로 '안 본 상태'의 예측 확률을 구함
  - `metrics_from(...)`: 그 확률로 점수 계산 → `rows`에 저장, 화면에도 출력
- 마지막에 결과를 표(res)로 만들어 CSV 저장. `proba_store`는 다음 셀(그래프)에서 재사용.

In [ ]:
# 3종 특징 집합을 같은 조건(5-fold CV, RandomForest)으로 비교
def make_model():                                        # 모델을 만들어 돌려주는 함수
    return RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                  random_state=42, n_jobs=-1)   # ← 괄호 안이 하이퍼파라미터(튜닝 대상)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 클래스 비율 유지한 5조각
rows, proba_store = [], {}                                # rows=점수 모음, proba_store=확률 보관(그래프용)
for name, X in FEATURES.items():                          # 입력 3종을 하나씩
    proba = cross_val_predict(make_model(), X, y, cv=skf, # 교차검증 예측 확률(안 본 상태의 예측)
                              method='predict_proba', n_jobs=-1)[:, 1]  # [:,1]=active(1)일 확률
    proba_store[name] = proba                             # 나중에 곡선 그릴 때 쓰려고 저장
    m = metrics_from(y, proba)                            # 점수 계산
    m = {'features': name, **m}                           # 결과에 입력 이름도 함께 담기
    rows.append(m)
    print('[%s] ROC-AUC %.3f | PR-AUC %.3f | MCC %.3f | Acc %.3f | Recall %.3f | Prec %.3f'
          % (name, m['ROC_AUC'], m['PR_AUC'], m['MCC'], m['Accuracy'], m['Recall'], m['Precision']))

res = pd.DataFrame(rows)                                  # 점수들을 표로
OUT = 'data/HSD17B13_model_eval_1to1.csv'
res.to_csv(OUT, index=False)                              # 결과 표 저장
print('\n결과 저장:', OUT)
print(res[['features', 'ROC_AUC', 'PR_AUC', 'MCC', 'Accuracy', 'Recall', 'Precision']]
      .round(3).to_string(index=False))

### 셀 5 — ROC · PR 곡선 그리기
그림을 좌우 두 칸으로 만들어(`subplots`), 입력 3종의 곡선을 겹쳐 그린다.
- **ROC 곡선**: 임계값을 바꿔가며 TPR(=Recall) vs FPR. 좌상단에 붙을수록 좋음(대각 점선=무작위).
- **PR 곡선**: Recall vs Precision. 불균형 데이터에서 특히 유용.
`savefig`로 PNG 저장, `show`로 노트북에 표시.

In [ ]:
# ROC / PR 커브 (3종 비교)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))            # 그림 1행 2칸 (왼=ROC, 오=PR)
for name, proba in proba_store.items():                  # 입력 3종의 확률을 하나씩
    fpr, tpr, _ = roc_curve(y, proba)                    # ROC 곡선 좌표 계산
    ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (name, roc_auc_score(y, proba)))  # 왼쪽에 그림
    pr, rc, _ = precision_recall_curve(y, proba)         # PR 곡선 좌표 계산
    ax[1].plot(rc, pr, label='%s (PR-AUC=%.3f)' % (name, average_precision_score(y, proba)))  # 오른쪽에
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')  # 무작위 기준선
ax[0].set_title('ROC curve'); ax[0].legend(loc='lower right', fontsize=9)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall curve'); ax[1].legend(loc='lower left', fontsize=9)
plt.tight_layout()
FIG = 'data/HSD17B13_model_eval_1to1.png'
plt.savefig(FIG, dpi=130)                                # 그림을 파일로 저장
plt.show()                                               # 노트북에 그림 표시
print('그림 저장:', FIG)